In [ ]:
# -*- coding: utf-8 -*-
"""
🎨 코요 캡션 분석기: 메타데이터로 AI의 눈을 훈련하는 실습 🌟

✅ 데이터셋 이름: kakaobrain/coyo-700m
✅ 데이터의 의미: 코요(COYO)는 이미지와 텍스트가 쌍을 이루는 (Image-Text Pair) 대규모 데이터셋입니다.
✅ 이 스크립트의 목표: 우리는 이 데이터셋의 수많은 정보(메타데이터)를 활용하여,
  단순히 '그림에 대한 설명'을 넘어, '이 설명이 얼마나 좋은지'를 AI가 분석하는 방법을 배웁니다.
  (핵심: 시각적 정보와 수치적 정보(Similarity Score)를 결합하여 효율성을 분석합니다.)

🚀 난이도: ⭐ (파이썬 기초, 데이터 분석 개념 맛보기)
💡 사용 개념: 데이터셋 로딩, 스트리밍 처리, 통계적 분석, 조건문 활용
"""
import random
import numpy as np
from datasets import load_dataset, Features, Value, ClassLabel

# --- [설정 값] ---
DATASET_ID = "kakaobrain/coyo-700m"
SAMPLE_SPLIT = 'train'
SAMPLE_COUNT = 100  # 너무 많은 샘플은 느려지므로, 분석을 위해 100개만 가져와요!
# -----------------

print("=" * 80)
print("📚 튜터가 알려주는 COYO-700M 데이터 분석 실습을 시작합니다! 📚")
print("=" * 80)

# 1. 데이터셋 로드 (스트리밍 처리 방어막)
print(f"\n[Step 1/4] 데이터셋 '{DATASET_ID}' 로드를 시도합니다...")
dataset = None
try:
    # 🌟 우선 스트리밍 모드로 로드하는 것이 가장 빠르고 메모리 효율적입니다.
    dataset = load_dataset(DATASET_ID, split=SAMPLE_SPLIT, streaming=True)
    print("✨ 성공! 스트리밍(Streaming) 방식으로 데이터 로딩 준비 완료!")
except Exception as e:
    # 🚨 만약 스트리밍 모드가 특정 환경에서 실패할 경우를 대비한 안전장치예요!
    print(f"⚠️ 스트리밍 로드 실패 감지: {e} (아마도 환경 문제일 수 있어요).")
    try:
        dataset = load_dataset(DATASET_ID, split=SAMPLE_SPLIT, streaming=False)
        print("✅ Fallback! 일반 모드(Non-streaming)로 데이터 로드 준비 완료! (시간이 좀 걸릴 수 있어요)")
    except Exception as e_fallback:
        print(f"🛑 에러가 발생했습니다. 데이터셋을 로드할 수 없습니다: {e_fallback}")
        exit()


# 2. 샘플 데이터 추출 (가장 효율적인 방법 사용)
print(f"\n[Step 2/4] {SAMPLE_COUNT}개의 대표 샘플을 추출합니다...")

# 🚀 스트리밍 모드와 일반 모드에 관계없이 안전하게 샘플 리스트를 확보하는 마법 패턴!
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)이므로, 리스트로 변환하여 사용합니다.
    sampled_dataset_iterator = dataset.take(SAMPLE_COUNT)
    sample_data_list = list(sampled_dataset_iterator)
    print(f"✅ {len(sample_data_list)}개의 샘플 데이터를 준비했습니다. 좋아요!")
else:
    # 일반 데이터셋 (Dataset)인 경우
    sample_data_list = list(dataset.select(range(min(SAMPLE_COUNT, len(dataset)))))
    print(f"✅ {len(sample_data_list)}개의 샘플 데이터를 준비했습니다. 좋아요!")


# 3. 데이터 구조 탐색 및 분석 (메타데이터가 곧 금광!)
print("\n" + "=" * 80)
print("[Step 3/4] 데이터셋의 구조와 주요 특징을 분석해봅시다!")
print("=" * 80)

# 첫 번째 샘플을 보고 데이터의 구조를 파악해봅니다.
first_sample = sample_data_list[0]

print("\n📘 튜터 분석: 이 데이터셋은 이미지와 함께 매우 많은 '점수'를 가지고 있어요.")
print("   - 'clip_similarity_vitl14': CLIP 모델을 이용한 유사도 점수 (Float). 이 점수가 높을수록 설명이 이미지를 잘 설명한다는 뜻!")
print("   - 'word_count': 캡션의 길이 (글자 수).")
print("   - 'aesthetic_score_laion_v2': 이미지 자체의 미적 점수.")


# --- [🧠 핵심 실습 1: 통계적 탐색] ---
print("\n🔎 분석 1: 평균 '좋은' 점수 찾아보기 (CLIP 유사도)")

# 모든 샘플에서 '클립 유사도' 점수만 모아서 평균을 계산해봅시다.
# 이 점수가 높을수록, 해당 캡션이 이미지의 내용을 잘 포착했다는 증거입니다!
clip_scores = [sample.get('clip_similarity_vitl14', 0.0) for sample in sample_data_list]
average_clip_score = np.mean(clip_scores)

print(f"   ⭐ {len(clip_scores)}개 샘플의 평균 CLIP 유사도 점수: {average_clip_score:.4f}")
print("   💡 해석: 이 점수가 높을수록 데이터셋이 매우 품질 좋은 캡션을 포함하고 있다는 뜻이에요!")


# --- [🎯 핵심 실습 2: 창의적인 필터링 (AI의 판단 시뮬레이션)] ---
print("\n🎯 분석 2: '최고의 조합' 샘플을 찾아봅시다. (똑똑한 AI처럼!)")
print("   - 목표: '매우 높은 유사도 점수'와 '적절한 길이의 캡션'을 가진 샘플을 찾는다.")

# 조건을 설정해봅시다.
TARGET_CLIP_THRESHOLD = 0.75  # 유사도가 0.75 이상인 샘플만 골라요.
MIN_WORD_COUNT = 3           # 글자 수가 3개 이상인 샘플만 골라요.

excellent_captions = []
for sample in sample_data_list:
    clip = sample.get('clip_similarity_vitl14', 0.0)
    word_count = sample.get('word_count', 0)

    # 🤖 조건 만족 여부 체크! (높은 유사도 AND 적당한 길이)
    if clip >= TARGET_CLIP_THRESHOLD and word_count >= MIN_WORD_COUNT:
        excellent_captions.append(sample)

print(f"\n   ✨ 조건을 만족하는 '최고의 캡션' 후보를 {len(excellent_captions)}개 발견했습니다!")

# 가장 좋은 후보 3개의 메타데이터를 보여줍니다.
print("\n--- TOP 3 BEST CANDIDATE METADATA ---")
for i, candidate in enumerate(excellent_captions[:3]):
    print(f"\n[✨ 후보 {i+1} ✨]")
    print(f"  🖼️ 이미지 정보: {candidate.get('width')}x{candidate.get('height')} px")
    print(f"  📝 캡션 (Text): {candidate.get('text')[:60]}...")
    print(f"  ✅ 유사도 점수 (CLIP): {candidate.get('clip_similarity_vitl14'):.4f}")
    print(f"  ✨ 품질 등급: '매우 좋음' (유사도 > {TARGET_CLIP_THRESHOLD:.2f})")


# 4. 마무리 및 정리
print("\n" + "=" * 80)
print("🎉 실습 완료! 정말 대단해요! 🎉")
print("=" * 80)
print("✨ 여러분은 단순한 데이터를 보는 것이 아니라, AI 모델이 학습하는 '품질 기준'을 스스로 세워봤어요.")
print("✨ 이제 어떤 AI 모델을 만날지 기대해도 좋아요!")